In [3]:
!pip -q install gradio timm

In [4]:
import json, cv2, torch, timm, gradio as gr
import numpy as np
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from google.colab import drive

MODEL_PATH = "/content/drive/MyDrive/ViT-B-16 Model Pth/vit_b16_location_deployment.pth"

CONFIG_PATH = "/content/drive/MyDrive/ViT-B-16 Model Pth/vit_b16_location_deployment_config.json"

In [5]:
class ViTLocationClassifier(nn.Module):
    def __init__(self, meta_dim, num_classes):
        super().__init__()
        self.backbone = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)
        self.meta_net = nn.Sequential(nn.Linear(meta_dim, 32), nn.ReLU(), nn.Dropout(0.2))
        self.head = nn.Sequential(
            nn.LayerNorm(self.backbone.num_features + 32),
            nn.Dropout(0.4),
            nn.Linear(self.backbone.num_features + 32, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, meta):
        img_feat = self.backbone(image)
        meta_feat = self.meta_net(meta)
        return self.head(torch.cat([img_feat, meta_feat], dim=1))

In [6]:
def remove_hair(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    clean = cv2.inpaint(bgr, mask, 1, cv2.INPAINT_TELEA)
    return cv2.cvtColor(clean, cv2.COLOR_BGR2RGB)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

class_names = config["class_names"]
meta_features = config["meta_features"]
locations = [x.replace("loc_", "") for x in meta_features]

In [8]:
model = ViTLocationClassifier(config["meta_dim"], config["num_classes"]).to(device)

checkpoint = torch.load(MODEL_PATH, map_location=device)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

ViTLocationClassifier(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (norm): Identity()
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)


In [9]:
transform = transforms.Compose([
    transforms.Resize((config["img_size"], config["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [10]:
def predict(image, location):
    if image is None:
        return {"Please upload image": 1.0}

    if location is None:
        return {"Please select location": 1.0}

    image = np.array(image)

    if config.get("hair_removal", False):
        image = remove_hair(image)

    image = Image.fromarray(image)
    image_tensor = transform(image).unsqueeze(0).to(device)

    location_key = "loc_" + location

    if location_key not in meta_features:
        return {"Invalid location selected": 1.0}

    meta = np.zeros(len(meta_features), dtype=np.float32)
    meta[meta_features.index(location_key)] = 1.0
    meta_tensor = torch.tensor(meta).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image_tensor, meta_tensor)
        probs = torch.softmax(output, dim=1)[0]

    result = {}
    for i in range(len(class_names)):
        result[class_names[i]] = float(probs[i])

    return result

In [11]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="pil", label="Upload Skin Image"),
        gr.Dropdown(
            choices=locations,
            value=locations[0],
            label="Select Location"
        )
    ],
    outputs=gr.Label(num_top_classes=7, label="Model Prediction"),
    title="Skin Disease Prediction",
    description="Upload skin image, select body location, then click Submit."
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ca4192db91d8d92d58.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
